# Bitcoin Price Prediction — Reproducible Benchmark (v16)

Companion notebook for *"Evaluating Machine Learning Models for Bitcoin Price
Prediction: A Reproducible Benchmark with Formal Statistical Testing and
Transaction-Cost Sensitivity"* (SN Computer Science).

Run the cells top to bottom. Every table and figure in the manuscript is
produced here; the final cell asserts that the figures agree with the tables.

**Layout expected by this notebook**

```
btc-prediction-benchmark/
├── data/       btc_features_final.csv        (input)
├── notebooks/  this file
├── figures/    *.png                         (generated)
└── results/    *.csv, manifest.json          (generated)
```

Paths resolve automatically from the notebook location. To override, set the
environment variable `BTC_BENCHMARK_ROOT` before launching Jupyter.

**Changes from v15**

1. Paths are portable — no absolute Windows path.
2. `fig3_dm_forest.png` marks the Bonferroni-adjusted threshold |z| = 2.64
   (α = 0.05/6 = 0.0083), which is the level the manuscript tests at, not the
   unadjusted |z| = 1.96.
3. The figure-rename step now **overwrites** its targets. In v15 it skipped any
   file that already existed, so stale PNGs from earlier runs survived into the
   LaTeX build — this is what caused figures to disagree with tables.
4. The per-regime table prints all seven models instead of four. The
   computation was always complete; only the display was truncated.
5. The final summary computes its numbers instead of hard-coding them, and
   reports the resolution of the PT test.
6. Phase 6 (split-conformal) is retained but **flagged as not reported**: its
   calibration set is in-sample. See the note above that phase.
7. A closing verification cell cross-checks figures against tables, confirms
   that per-regime RMSEs pool back to the overall test RMSE, and lists any
   manuscript figure that is missing.


## Phase 1 — Setup, imports, global configuration

In [ ]:
# Standard library
import os
import time
import warnings
import json
from pathlib import Path
from itertools import product

# Suppress TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
warnings.filterwarnings('ignore')

# Scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

# Boosting
import xgboost as xgb

# Statistics
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss

# Interpretability
import shap

# Deep learning
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout,
                                      LSTM, GRU, Dense, Layer)
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow.keras.backend as K

# Display settings
pd.set_option('display.float_format', '{:,.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
import random
random.seed(RANDOM_SEED)

# Paths -- resolved relative to the repository, overridable via BTC_BENCHMARK_ROOT
_env = os.environ.get('BTC_BENCHMARK_ROOT')
if _env:
    ROOT = Path(_env).expanduser().resolve()
else:
    _here = Path.cwd().resolve()
    ROOT = _here.parent if _here.name == 'notebooks' else _here
DATA_DIR = ROOT / 'data'
FIG_DIR = ROOT / 'figures'
RES_DIR = ROOT / 'results'
for _d in (DATA_DIR, FIG_DIR, RES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

FEATURES_CSV = DATA_DIR / 'btc_features_final.csv'
print(f'Repository root : {ROOT}')
print(f'Input CSV found : {FEATURES_CSV.exists()}  ({FEATURES_CSV})')
if not FEATURES_CSV.exists():
    print('  -> place btc_features_final.csv in data/, or set BTC_BENCHMARK_ROOT')

print('Environment configured')
print(f'  numpy   : {np.__version__}')
print(f'  pandas  : {pd.__version__}')
print(f'  xgboost : {xgb.__version__}')
print(f'  tensorflow : {tf.__version__}')
print(f'  shap    : {shap.__version__}')
print(f'  Random seed: {RANDOM_SEED}')


### Load the feature dataset

The CSV `btc_features_final.csv` was constructed from CoinMarketCap OHLCV data
(2017-01-01 → 2023-12-31) plus Google Trends (monthly, linearly interpolated to
daily). Feature engineering produces 22 columns. After rolling-window computations,
2{,}525 daily observations remain with no missing values.

In [ ]:
# Load main feature dataset
df = pd.read_csv(FEATURES_CSV, parse_dates=['date'])
df = df.set_index('date').sort_index()

print(f'Shape: {df.shape[0]} rows × {df.shape[1]} cols')
print(f'Date range: {df.index.min().date()} → {df.index.max().date()}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nColumns: {df.columns.tolist()}')

# Define feature set (20 features) and target
FEATURE_COLS = [
    'open', 'high', 'low', 'close', 'volume', 'market_cap',
    'google_trends', 'log_return',
    'volatility_7d', 'volatility_30d',
    'rsi_14', 'ema_7', 'ema_30',
    'macd', 'macd_signal',
    'bb_upper', 'bb_lower', 'bb_width',
    'high_low_ratio', 'close_open_ratio'
]
TARGET_COL = 'target_price'

print(f'\n{len(FEATURE_COLS)} features, target = {TARGET_COL!r}')
df[FEATURE_COLS + [TARGET_COL]].head(3)


### Table 1 — Descriptive statistics of principal variables

In [ ]:
variables_show = ['close', 'log_return', 'volatility_7d', 'rsi_14', 'google_trends']
desc = df[variables_show].describe().T[['mean', 'std', 'min', 'max']]
desc.columns = ['Mean', 'Std', 'Min', 'Max']
print('Table 1 — Descriptive statistics')
print(desc.round(4))
desc.to_csv(RES_DIR / 'table1_descriptive_stats.csv')


### Table 4 — Stationarity diagnostics

ADF and KPSS tests confirm that **close prices are non-stationary** (level
form is I(1)) while **log returns are stationary**.

In [ ]:
def adf_kpss_pair(series, name):
    s = series.dropna()
    adf_stat, adf_p = adfuller(s)[:2]
    try:
        kpss_stat, kpss_p = kpss(s, regression='c', nlags='auto')[:2]
        # KPSS returns the closest p-value in table
        kpss_p_str = f'<{kpss_p:.2f}' if kpss_p <= 0.01 else (f'>{kpss_p:.2f}' if kpss_p >= 0.1 else f'{kpss_p:.3f}')
    except Exception:
        kpss_stat, kpss_p_str = float('nan'), 'n/a'
    return {'series': name, 'adf_stat': adf_stat, 'adf_p': adf_p,
            'kpss_stat': kpss_stat, 'kpss_p': kpss_p_str}

stat = pd.DataFrame([
    adf_kpss_pair(df['close'], 'Close (levels)'),
    adf_kpss_pair(df['log_return'].dropna(), 'Log return'),
])
print('Table 4 — Stationarity tests')
print(stat.to_string(index=False))
stat.to_csv(RES_DIR / 'table4_stationarity.csv', index=False)


## Phase 2 — Baseline models (Random Walk, Random Forest, XGBoost)

### Temporal train/validation/test split with strict leakage prevention

- **Train**: Jan 2017 → Dec 2020 (1431 obs) — for initial training & hyperparameter search
- **Validation**: Jan 2021 → Dec 2021 (365 obs) — for hyperparameter selection
- **Test**: Jan 2022 → Dec 2023 (729 obs) — held-out for final evaluation

The `StandardScaler` is fitted on the training split only. After hyperparameter
selection, the final model is retrained on Train ∪ Validation.

In [ ]:
TRAIN_END = '2020-12-31'
VAL_END = '2021-12-31'
TEST_END = '2023-12-30'

train = df.loc[:TRAIN_END]
val = df.loc[pd.Timestamp(TRAIN_END) + pd.Timedelta(days=1):VAL_END]
test = df.loc[pd.Timestamp(VAL_END) + pd.Timedelta(days=1):TEST_END]

print('Table 5 — Temporal splits')
print(f'  Train: {len(train):>4} obs ({train.index.min().date()} → {train.index.max().date()})')
print(f'  Val:   {len(val):>4} obs ({val.index.min().date()} → {val.index.max().date()})')
print(f'  Test:  {len(test):>4} obs ({test.index.min().date()} → {test.index.max().date()})')
print(f'\nMean BTC train price: ${train["close"].mean():,.0f}')
print(f'Mean BTC test price:  ${test["close"].mean():,.0f}')
print(f'Ratio test/train:     {test["close"].mean() / train["close"].mean():.2f}×')

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(train[FEATURE_COLS])
X_val = scaler.transform(val[FEATURE_COLS])
X_test = scaler.transform(test[FEATURE_COLS])

y_train = train[TARGET_COL].values
y_val = val[TARGET_COL].values
y_test = test[TARGET_COL].values

# Combine train + val for final retraining
X_trainval = np.vstack([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])


### Evaluation metric helper

Computes RMSE, MAE, MAPE, R², and Directional Accuracy. Ties in first-difference
sign (when Δy = 0 or Δŷ = 0) are counted as **incorrect predictions**, consistent
with the manuscript's footnote. On our test set, ties account for < 0.3% of
observations.

In [ ]:
def compute_metrics(y_true, y_pred, name='model'):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    e = y_true - y_pred
    rmse = float(np.sqrt((e**2).mean()))
    mae = float(np.abs(e).mean())
    mape = float((np.abs(e / y_true).mean()) * 100)
    r2 = float(1 - (e**2).sum() / ((y_true - y_true.mean())**2).sum())

    dy = np.diff(y_true)
    dp = np.diff(y_pred)
    # Ties (sign==0 on either side) counted as incorrect
    correct = (np.sign(dy) == np.sign(dp)) & (dy != 0) & (dp != 0)
    da = float(correct.mean() * 100)
    n_ties = int(((dy == 0) | (dp == 0)).sum())

    return {'model': name, 'rmse': rmse, 'mae': mae, 'mape': mape,
            'r2': r2, 'da': da, 'n_ties': n_ties}


# Container for all model predictions and errors
predictions = {}    # name -> np.array of test-set predictions
errors = {}         # name -> np.array of squared errors
all_metrics = []


### Model 1 — Random Walk (naive baseline)

$\hat{P}_{t+1} = P_t$. This is equivalent to ARIMA(0,1,0) and corresponds to the
efficient-market null hypothesis. Note that RW's apparent dominance on RMSE/R² is
a mechanical consequence of high daily price autocorrelation, not predictive skill.

In [ ]:
# Random Walk: predict P[t+1] as today's close
rw_pred = test['close'].values
predictions['Random Walk'] = rw_pred
errors['Random Walk'] = (y_test - rw_pred)**2
m = compute_metrics(y_test, rw_pred, 'Random Walk')
all_metrics.append(m)
print(f'Random Walk: RMSE=${m["rmse"]:,.0f}, MAE=${m["mae"]:,.0f}, '
      f'MAPE={m["mape"]:.2f}%, R²={m["r2"]:.3f}, DA={m["da"]:.2f}%')


### Model 2 — Random Forest

Ensemble of 100 decision trees with feature subsampling (√d) and bootstrap
sampling. Max depth = 10. Trained on Train ∪ Val.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100, max_depth=10, max_features='sqrt',
    bootstrap=True, random_state=RANDOM_SEED, n_jobs=-1
)
rf.fit(X_trainval, y_trainval)
rf_pred = rf.predict(X_test)
predictions['Random Forest'] = rf_pred
errors['Random Forest'] = (y_test - rf_pred)**2
m = compute_metrics(y_test, rf_pred, 'Random Forest')
all_metrics.append(m)
print(f'Random Forest: RMSE=${m["rmse"]:,.0f}, MAE=${m["mae"]:,.0f}, '
      f'MAPE={m["mape"]:.2f}%, R²={m["r2"]:.3f}, DA={m["da"]:.2f}%')


### Model 3 — XGBoost with grid search

Exhaustive grid over 27 configurations:
- learning_rate ∈ {0.05, 0.1, 0.2}
- max_depth ∈ {4, 6, 8}
- n_estimators ∈ {200, 500, 1000}

Selection criterion: validation-set RMSE. Best configuration is retrained on
Train ∪ Val.

In [ ]:
param_grid = {
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [4, 6, 8],
    'n_estimators': [200, 500, 1000]
}

grid_results = []
for lr, depth, n_est in product(*param_grid.values()):
    m = xgb.XGBRegressor(learning_rate=lr, max_depth=depth, n_estimators=n_est,
                         random_state=RANDOM_SEED, n_jobs=-1, verbosity=0)
    m.fit(X_train, y_train)
    val_rmse = float(np.sqrt(((y_val - m.predict(X_val))**2).mean()))
    grid_results.append({'lr': lr, 'depth': depth, 'n_est': n_est, 'val_rmse': val_rmse})

grid_df = pd.DataFrame(grid_results).sort_values('val_rmse').reset_index(drop=True)
best = grid_df.iloc[0]
print(f'Best XGBoost configuration:')
print(f'  learning_rate = {best["lr"]}')
print(f'  max_depth     = {int(best["depth"])}')
print(f'  n_estimators  = {int(best["n_est"])}')
print(f'  val_RMSE      = ${best["val_rmse"]:,.0f}')
grid_df.to_csv(RES_DIR / 'xgb_grid_search.csv', index=False)
grid_df.head(10)


In [ ]:
# Retrain XGBoost with best config on train ∪ val
xgb_final = xgb.XGBRegressor(
    learning_rate=best['lr'],
    max_depth=int(best['depth']),
    n_estimators=int(best['n_est']),
    random_state=RANDOM_SEED, n_jobs=-1, verbosity=0
)
xgb_final.fit(X_trainval, y_trainval)
xgb_pred = xgb_final.predict(X_test)
predictions['XGBoost'] = xgb_pred
errors['XGBoost'] = (y_test - xgb_pred)**2
m = compute_metrics(y_test, xgb_pred, 'XGBoost')
all_metrics.append(m)
print(f'XGBoost: RMSE=${m["rmse"]:,.0f}, MAE=${m["mae"]:,.0f}, '
      f'MAPE={m["mape"]:.2f}%, R²={m["r2"]:.3f}, DA={m["da"]:.2f}%')


## Phase 3 — Deep learning models

### Bahdanau attention layer and seeding helper

Deep learning models use a sliding window of L=30 past observations. Target is
**scaled** to stabilise training, then inverse-transformed back to USD for
evaluation. Each model is trained over 3 independent seeds (42, 123, 456) and
final predictions are the mean across seeds, with pointwise standard deviation
reported.

In [ ]:
LOOKBACK = 30
SEEDS = [42, 123, 456]

def set_seeds(seed):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    random.seed(seed)

def to_sequences(X, y, lookback=LOOKBACK):
    Xs, ys = [], []
    for i in range(len(X) - lookback):
        Xs.append(X[i:i+lookback])
        ys.append(y[i+lookback])
    return np.array(Xs), np.array(ys)

class BahdanauAttention(Layer):
    """Bahdanau-style additive attention."""
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.W = Dense(units)
        self.v = Dense(1)
    def call(self, h):
        score = self.v(tf.nn.tanh(self.W(h)))
        alpha = tf.nn.softmax(score, axis=1)
        return tf.reduce_sum(alpha * h, axis=1)

# Target scaler (separate from feature scaler)
target_scaler = StandardScaler()
y_train_s = target_scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
y_val_s = target_scaler.transform(y_val.reshape(-1, 1)).flatten()
y_trainval_s = np.concatenate([y_train_s, y_val_s])

# Build sequences from train ∪ val
X_trainval_seq, y_trainval_seq = to_sequences(X_trainval, y_trainval_s)

# Test sequences need LOOKBACK observations from validation to seed them
X_test_input = np.vstack([X_val[-LOOKBACK:], X_test])
y_test_input = np.concatenate([y_val[-LOOKBACK:], y_test])
y_test_input_s = target_scaler.transform(y_test_input.reshape(-1, 1)).flatten()
X_test_seq, _ = to_sequences(X_test_input, y_test_input_s)
print(f'Train sequences: {X_trainval_seq.shape}')
print(f'Test sequences:  {X_test_seq.shape}')
assert X_test_seq.shape[0] == len(y_test), 'Sequence count must match test length'


### Model 4 — CNN-LSTM hybrid

Conv1D (64 filters, kernel=3) → MaxPool(2) → Dropout(0.2) → LSTM(50) → Dropout(0.2) → Dense(1).
≈35,700 trainable parameters.

In [ ]:
def build_cnn_lstm(input_shape):
    K.clear_session()
    inp = Input(shape=input_shape)
    x = Conv1D(64, 3, activation='relu')(inp)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.2)(x)
    x = LSTM(50)(x)
    x = Dropout(0.2)(x)
    out = Dense(1)(x)
    model = Model(inp, out)
    model.compile(optimizer='adam', loss='mse')
    return model

cnn_preds = []
cnn_times = []
for seed in SEEDS:
    set_seeds(seed)
    t0 = time.time()
    m = build_cnn_lstm((LOOKBACK, len(FEATURE_COLS)))
    m.fit(X_trainval_seq, y_trainval_seq, epochs=100, batch_size=32, verbose=0,
          callbacks=[EarlyStopping(patience=15, restore_best_weights=True,
                                    monitor='loss')])
    pred_s = m.predict(X_test_seq, verbose=0).flatten()
    pred = target_scaler.inverse_transform(pred_s.reshape(-1, 1)).flatten()
    cnn_preds.append(pred)
    cnn_times.append(time.time() - t0)
    print(f'   CNN-LSTM seed {seed}: trained in {cnn_times[-1]:.1f}s')

cnn_preds_a = np.array(cnn_preds)
cnn_mean = cnn_preds_a.mean(axis=0)
cnn_pointwise_std = cnn_preds_a.std(axis=0).mean()

predictions['CNN-LSTM'] = cnn_mean
errors['CNN-LSTM'] = (y_test - cnn_mean)**2
m_cnn = compute_metrics(y_test, cnn_mean, 'CNN-LSTM')
all_metrics.append(m_cnn)

print(f'\nCNN-LSTM: mean RMSE=${m_cnn["rmse"]:,.0f}, '
      f'pointwise std=${cnn_pointwise_std:,.0f} '
      f'({100*cnn_pointwise_std/m_cnn["rmse"]:.1f}% of mean RMSE)')
print(f'           DA={m_cnn["da"]:.2f}%, avg training {np.mean(cnn_times):.1f}s/seed')


### Model 5 — GRU + Bahdanau Attention

GRU(64, return_sequences=True) → Bahdanau attention → Dense(1).
≈25,700 trainable parameters.

In [ ]:
def build_gru_attention(input_shape):
    K.clear_session()
    inp = Input(shape=input_shape)
    h = GRU(64, return_sequences=True)(inp)
    c = BahdanauAttention(64)(h)
    out = Dense(1)(c)
    model = Model(inp, out)
    model.compile(optimizer='adam', loss='mse')
    return model

gru_preds = []
gru_times = []
for seed in SEEDS:
    set_seeds(seed)
    t0 = time.time()
    m = build_gru_attention((LOOKBACK, len(FEATURE_COLS)))
    m.fit(X_trainval_seq, y_trainval_seq, epochs=100, batch_size=32, verbose=0,
          callbacks=[EarlyStopping(patience=15, restore_best_weights=True,
                                    monitor='loss')])
    pred_s = m.predict(X_test_seq, verbose=0).flatten()
    pred = target_scaler.inverse_transform(pred_s.reshape(-1, 1)).flatten()
    gru_preds.append(pred)
    gru_times.append(time.time() - t0)
    print(f'   GRU+Att seed {seed}: trained in {gru_times[-1]:.1f}s')

gru_preds_a = np.array(gru_preds)
gru_mean = gru_preds_a.mean(axis=0)
gru_pointwise_std = gru_preds_a.std(axis=0).mean()

predictions['GRU+Att'] = gru_mean
errors['GRU+Att'] = (y_test - gru_mean)**2
m_gru = compute_metrics(y_test, gru_mean, 'GRU+Att')
all_metrics.append(m_gru)

print(f'\nGRU+Att: mean RMSE=${m_gru["rmse"]:,.0f}, '
      f'pointwise std=${gru_pointwise_std:,.0f} '
      f'({100*gru_pointwise_std/m_gru["rmse"]:.1f}% of mean RMSE)')
print(f'         DA={m_gru["da"]:.2f}%, avg training {np.mean(gru_times):.1f}s/seed')


### Model 6-7 — GRU+Att Walk-Forward (90d and 180d)

Walk-forward retraining: model retrained from scratch every 90 (or 180) days
using all data available up to that point. Single-seed per segment to keep
retraining costs tractable.

In [ ]:
def walk_forward_gru(retrain_days, lookback=LOOKBACK, seed=42):
    """
    Walk-forward: retrain GRU+Att every `retrain_days` days using all data
    available up to that point. Returns predictions for the entire test set.
    """
    set_seeds(seed)
    full_X = np.vstack([X_trainval, X_test])
    full_y = np.concatenate([y_trainval_s,
                             target_scaler.transform(y_test.reshape(-1, 1)).flatten()])
    test_start_idx = len(X_trainval)

    predictions_wf = np.zeros(len(X_test))
    n_segments = int(np.ceil(len(X_test) / retrain_days))

    for seg in range(n_segments):
        seg_start = seg * retrain_days
        seg_end = min(seg_start + retrain_days, len(X_test))

        # Training data = everything before this segment starts
        train_end_idx = test_start_idx + seg_start
        Xtr = full_X[:train_end_idx]
        ytr = full_y[:train_end_idx]
        Xtr_seq, ytr_seq = to_sequences(Xtr, ytr, lookback)

        # Build & train
        model = build_gru_attention((lookback, len(FEATURE_COLS)))
        model.fit(Xtr_seq, ytr_seq, epochs=100, batch_size=32, verbose=0,
                  callbacks=[EarlyStopping(patience=15, restore_best_weights=True,
                                            monitor='loss')])

        # Predict on this segment
        seg_X_input = full_X[train_end_idx - lookback:test_start_idx + seg_end]
        seg_y_input = full_y[train_end_idx - lookback:test_start_idx + seg_end]
        Xte_seq, _ = to_sequences(seg_X_input, seg_y_input, lookback)
        pred_s = model.predict(Xte_seq, verbose=0).flatten()
        pred = target_scaler.inverse_transform(pred_s.reshape(-1, 1)).flatten()
        predictions_wf[seg_start:seg_end] = pred

    return predictions_wf

print('GRU+Att Walk-Forward 90d (9 segments)...')
t0 = time.time()
wf90_pred = walk_forward_gru(90, seed=42)
print(f'   Total time: {time.time()-t0:.1f}s')

predictions['GRU+Att WF 90d'] = wf90_pred
errors['GRU+Att WF 90d'] = (y_test - wf90_pred)**2
m_wf90 = compute_metrics(y_test, wf90_pred, 'GRU+Att WF 90d')
all_metrics.append(m_wf90)
print(f'GRU+Att WF 90d: RMSE=${m_wf90["rmse"]:,.0f}, DA={m_wf90["da"]:.2f}%')

print('\nGRU+Att Walk-Forward 180d (5 segments)...')
t0 = time.time()
wf180_pred = walk_forward_gru(180, seed=42)
print(f'   Total time: {time.time()-t0:.1f}s')

predictions['GRU+Att WF 180d'] = wf180_pred
errors['GRU+Att WF 180d'] = (y_test - wf180_pred)**2
m_wf180 = compute_metrics(y_test, wf180_pred, 'GRU+Att WF 180d')
all_metrics.append(m_wf180)
print(f'GRU+Att WF 180d: RMSE=${m_wf180["rmse"]:,.0f}, DA={m_wf180["da"]:.2f}%')


## Phase 4 — Statistical inference

### Diebold-Mariano test with HLN small-sample correction

For each challenger model vs XGBoost reference, test
$H_0: \mathbb{E}[d_t] = 0$ where $d_t = e^2_{i,t} - e^2_{\text{XGB},t}$.
Newey-West HAC variance (lag=1) is used; HLN correction adjusts for small samples.
Bonferroni correction across the 6 challengers: $\alpha / 6 = 0.0083$.

Note: XGBoost is selected as the reference **a priori** based on its expected
performance on tabular financial data, not post-hoc.

In [ ]:
def diebold_mariano_hln(e1_sq, e2_sq, h=1):
    """
    HLN-corrected DM test.
    e1_sq, e2_sq: squared errors of model 1 (challenger) and model 2 (reference XGBoost)
    Positive DM => XGBoost (e2) has lower expected squared loss.
    """
    d = e1_sq - e2_sq
    n = len(d)
    d_mean = d.mean()

    # Newey-West variance with lag = h-1 = 0 theoretically, but we use lag=1
    # as conservative choice for autocorrelation beyond one-step
    lag = max(h-1, 1)
    gamma = [np.mean((d - d_mean) * (d - d_mean))]
    for k in range(1, lag + 1):
        gamma_k = np.mean((d[k:] - d_mean) * (d[:-k] - d_mean))
        gamma.append(gamma_k)
    nw_var = gamma[0] + 2 * sum(gamma[1:])
    nw_var = max(nw_var, 1e-12)

    dm = d_mean / np.sqrt(nw_var / n)

    # HLN small-sample correction
    hln_factor = np.sqrt((n + 1 - 2*h + (h*(h-1))/n) / n)
    dm_hln = dm * hln_factor

    # Two-tailed p-value with t-distribution (HLN suggests t_{n-1})
    p_raw = 2 * (1 - stats.t.cdf(abs(dm_hln), df=n-1))
    return dm_hln, p_raw

# Run DM tests
challengers = ['Random Walk', 'Random Forest', 'CNN-LSTM', 'GRU+Att',
               'GRU+Att WF 90d', 'GRU+Att WF 180d']
e_xgb = errors['XGBoost']
dm_results = []
for ch in challengers:
    dm_stat, p_raw = diebold_mariano_hln(errors[ch], e_xgb)
    p_bonf = min(p_raw * 6, 1.0)
    if p_bonf < 0.01:
        winner = 'XGBoost wins' if dm_stat > 0 else f'{ch} wins'
    else:
        winner = 'No difference'
    dm_results.append({'Challenger': ch, 'DM stat': dm_stat,
                       'p (raw)': p_raw, 'p (Bonf.)': p_bonf,
                       'Conclusion': winner})

dm_df = pd.DataFrame(dm_results)
print('Table 8 — Diebold-Mariano tests vs XGBoost (HLN corrected, Bonferroni adjusted)')
print(dm_df.to_string(index=False, formatters={
    'DM stat': '{:+.2f}'.format,
    'p (raw)': '{:.3e}'.format,
    'p (Bonf.)': '{:.3e}'.format
}))
dm_df.to_csv(RES_DIR / 'table8_dm_tests.csv', index=False)


### Pesaran-Timmermann directional skill test

Non-parametric test of directional skill. Under $H_0$ of directional independence,
the realised hit rate equals the rate expected by chance.

In [ ]:
def pesaran_timmermann(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    dy = np.diff(y_true)
    dp = np.diff(y_pred)
    # Drop ties (sign==0)
    mask = (dy != 0) & (dp != 0)
    dy_s = np.sign(dy[mask])
    dp_s = np.sign(dp[mask])

    n = len(dy_s)
    P_hat = (dy_s == dp_s).mean()
    p_y = (dy_s > 0).mean()
    p_p = (dp_s > 0).mean()
    P_star = p_y * p_p + (1 - p_y) * (1 - p_p)

    # Variance under H0
    var_Phat = (P_star * (1 - P_star)) / n
    var_Pstar = (
        ((2*p_y - 1)**2) * p_p*(1-p_p) / n
        + ((2*p_p - 1)**2) * p_y*(1-p_y) / n
        + 4*p_y*p_p*(1-p_y)*(1-p_p) / n**2
    )
    var_diff = var_Phat - var_Pstar
    var_diff = max(var_diff, 1e-12)

    z = (P_hat - P_star) / np.sqrt(var_diff)
    p_val = 2 * (1 - stats.norm.cdf(abs(z)))
    return z, p_val

# PT for every model
pt_results = []
for name, pred in predictions.items():
    z, p = pesaran_timmermann(y_test, pred)
    pt_results.append({'Model': name, 'PT z': z, 'PT p': p,
                       'reject H0 at 5%': p < 0.05})

pt_df = pd.DataFrame(pt_results)
print('Table 9 — Pesaran-Timmermann directional skill tests')
print(pt_df.to_string(index=False, formatters={
    'PT z': '{:+.2f}'.format,
    'PT p': '{:.3f}'.format
}))
pt_df.to_csv(RES_DIR / 'table9_pesaran_timmermann.csv', index=False)

# Bonferroni at 5% / 7 = 0.0071
alpha_bonf = 0.05 / 7
print(f'\nBonferroni-adjusted alpha for 7 PT tests: {alpha_bonf:.4f}')
n_sig_raw = (pt_df['PT p'] < 0.05).sum()
n_sig_bonf = (pt_df['PT p'] < alpha_bonf).sum()
print(f'Models with raw p<0.05:       {n_sig_raw} of 7')
print(f'Models with Bonf-adj p<0.0071: {n_sig_bonf} of 7')
print('\n=> Directional Accuracy Pitfall: zero models show statistically')
print('   significant directional skill after Bonferroni correction.')


## Phase 5 -- Economic evaluation

### Long/short trading simulation with transaction-cost sensitivity

At the close of day $t$, take a long position if $\hat{P}_{t+1} > P_t$,
otherwise a short position. Daily strategy return:
$r_{s,t} = s_t \cdot (P_{t+1} - P_t)/P_t$ with $s_t \in \{-1, +1\}$.
Annualised Sharpe ratio $= \sqrt{252} \cdot \bar{r}_s / \sigma_s$.

**Caveats:**
- We omit the risk-free rate from the Sharpe numerator. For the 2022-2023
  test window, $R_f \approx 4\%$ would shift every figure by the same constant
  and would not alter the across-model ordering.
- Transaction costs are applied symmetrically on every side change at
  0 / 10 / 50 bp per side.
- Short-selling borrow costs (typically 5-15% annualised on major exchanges)
  are NOT modelled and would further depress the Sharpe of short-heavy strategies.

In [ ]:
def long_short_sharpe(P, y_pred, tc_bps=0, ann_factor=np.sqrt(252)):
    """
    P: actual close prices for the test set (used for realised returns)
    y_pred: model predictions of next-day price
    Returns annualised Sharpe at the given transaction cost (in bp per side).
    """
    P = np.asarray(P)
    # Signal: +1 long if predicting up, -1 short otherwise
    signal = np.where(y_pred > P, 1, -1)
    # Realised return next day
    next_P = np.append(P[1:], P[-1])  # use last price for last day
    r_real = (next_P - P) / P
    strategy_r = signal * r_real

    # Transaction cost on every side change (entering or exiting)
    side_changes = np.abs(np.diff(signal, prepend=signal[0]))  # 0 or 2
    tc = (tc_bps / 10000.0) * side_changes  # per side change
    strategy_r_net = strategy_r - tc

    sharpe = ann_factor * strategy_r_net.mean() / (strategy_r_net.std() + 1e-12)
    return sharpe, strategy_r_net

# Compute Sharpe at 3 cost levels for all models + buy-and-hold benchmark
P_test = test['close'].values
tc_levels = [0, 10, 50]
sharpe_table = []
strategy_returns = {}  # for cumulative plot

for name, pred in predictions.items():
    row = {'Model': name}
    for tc in tc_levels:
        s, r = long_short_sharpe(P_test, pred, tc)
        row[f'{tc} bp'] = s
        if tc == 0:
            strategy_returns[name] = r
    sharpe_table.append(row)

# Buy-and-hold
bh_r = (P_test[1:] - P_test[:-1]) / P_test[:-1]
bh_sharpe = np.sqrt(252) * bh_r.mean() / (bh_r.std() + 1e-12)
sharpe_table.append({'Model': 'Buy & Hold', '0 bp': bh_sharpe,
                     '10 bp': bh_sharpe, '50 bp': bh_sharpe})
strategy_returns['Buy & Hold'] = bh_r

sharpe_df = pd.DataFrame(sharpe_table)
print('Table 10 -- Sharpe ratio sensitivity to transaction costs')
print(sharpe_df.to_string(index=False, formatters={
    '0 bp': '{:+.3f}'.format,
    '10 bp': '{:+.3f}'.format,
    '50 bp': '{:+.3f}'.format
}))
sharpe_df.to_csv(RES_DIR / 'table10_sharpe_tc.csv', index=False)

# Add Sharpe(0bp) to all_metrics
for m in all_metrics:
    name = m['model']
    row = next((r for r in sharpe_table if r['Model'] == name), None)
    m['sharpe_0bp'] = row['0 bp'] if row else np.nan


In [ ]:
# Figure 7 -- Sharpe-vs-TC sensitivity (mirrors fig11_tc_sensitivity.png)
fig, ax = plt.subplots(figsize=(9, 5))
xpos = [0, 10, 50]

for name in predictions.keys():
    yvals = [sharpe_df.loc[sharpe_df['Model']==name, f'{tc} bp'].values[0] for tc in xpos]
    ax.plot(xpos, yvals, marker='o', linewidth=1.8, label=name)

# Buy & hold reference (horizontal at +0.135 or whatever computed)
ax.axhline(bh_sharpe, color='green', linestyle=':', linewidth=2,
           label=f'Buy & Hold ({bh_sharpe:+.3f})')
ax.axhline(0, color='gray', linestyle='-', linewidth=0.5)
ax.set_xlabel('Transaction cost per side (bp)')
ax.set_ylabel('Annualised Sharpe ratio')
ax.set_title('Sharpe-ratio sensitivity to transaction costs')
ax.legend(loc='best', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig7_tc_sensitivity.png', dpi=150)
plt.show()
print('\nObservation: at realistic 10 bp costs, no learned model robustly beats')
print(f'Buy & Hold (Sharpe = {bh_sharpe:+.3f}).')


In [ ]:
# Figure 6 -- Cumulative trading returns at 0 bp
fig, ax = plt.subplots(figsize=(10, 5))
for name, r in strategy_returns.items():
    cum = np.cumprod(1 + r) - 1
    ax.plot(test.index[:len(cum)], cum * 100,
            label=f'{name} ({cum[-1]*100:+.1f}%)', linewidth=1.5)

ax.set_xlabel('Date')
ax.set_ylabel('Cumulative return (%)')
ax.set_title('Long/short cumulative trading-strategy returns (0 bp)')
ax.legend(loc='best', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig6_trading_cumret.png', dpi=150)
plt.show()


### Figure 4 -- The Directional Accuracy Pitfall

DA vs Sharpe ratio across all 7 models. Spearman rank correlation is computed
to formally test the monotonicity claim.

In [ ]:
# Compute Spearman correlation DA vs Sharpe
da_values = [m['da'] for m in all_metrics]
sharpe_values = [m['sharpe_0bp'] for m in all_metrics]
spearman_r, spearman_p = stats.spearmanr(da_values, sharpe_values)

fig, ax = plt.subplots(figsize=(8, 6))
for m in all_metrics:
    ax.scatter(m['da'], m['sharpe_0bp'], s=180, alpha=0.75)
    ax.annotate(m['model'], (m['da'], m['sharpe_0bp']),
                xytext=(8, 8), textcoords='offset points', fontsize=10)

ax.axhline(0, color='gray', linestyle='-', linewidth=0.5)
ax.axvline(50, color='gray', linestyle='--', linewidth=0.5,
           label='Chance DA = 50%')
ax.axhline(bh_sharpe, color='green', linestyle=':', linewidth=1.5,
           label=f'B&H Sharpe = {bh_sharpe:+.3f}')

ax.set_xlabel('Directional Accuracy (%)')
ax.set_ylabel('Sharpe ratio (0 bp)')
ax.set_title(f'Directional Accuracy Pitfall: DA vs Sharpe ratio\n'
             f'(Spearman rho = {spearman_r:+.3f}, p = {spearman_p:.3f}, n = {len(all_metrics)})')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4_da_paradox.png', dpi=150)
plt.show()

print(f'\nSpearman rank correlation DA vs Sharpe: rho = {spearman_r:+.3f}, p = {spearman_p:.3f}')
print('No statistically significant monotonic association at the 5% level.')
print('This is the empirical foundation of the Directional Accuracy Pitfall.')


## Phase 6 — Split-conformal prediction intervals — **NOT REPORTED IN THE MANUSCRIPT**

> **This phase is retained for transparency but its output is not used in the paper.**
>
> The calibration set is **in-sample**. `xgb_final` is refitted on
> train ∪ validation, so `xgb_final.predict(X_val)` scores data the model has
> already seen. The residuals are memorised, the calibration quantile collapses
> to roughly \$70, and empirical coverage falls to about 3% against a 90%
> nominal level.
>
> Split-conformal requires a calibration set held out from training. Doing this
> correctly means fitting XGBoost on the training window only and keeping 2021
> as a genuine calibration set — a different experiment, not a patch. Until that
> is run, no conformal result should be quoted.


In [ ]:
# XGBoost validation residuals = calibration set
xgb_val_pred = xgb_final.predict(X_val)
val_residuals = np.abs(y_val - xgb_val_pred)

conformal_rows = []
for level in [0.90, 0.95]:
    q = np.quantile(val_residuals, level)
    lower = xgb_pred - q
    upper = xgb_pred + q
    cov = float(((y_test >= lower) & (y_test <= upper)).mean())
    width = float(2 * q)
    conformal_rows.append({'Level': f'{level*100:.0f}%',
                           'Nominal': level,
                           'Empirical coverage': cov,
                           'Mean width (USD)': width})

conf_df = pd.DataFrame(conformal_rows)
print('Table 11 -- Split-conformal intervals (XGBoost, validation calibration)')
print(conf_df.to_string(index=False, formatters={
    'Nominal': '{:.0%}'.format,
    'Empirical coverage': '{:.1%}'.format,
    'Mean width (USD)': '${:,.0f}'.format
}))
conf_df.to_csv(RES_DIR / 'table11_conformal.csv', index=False)


In [ ]:
# Figure 10 -- Conformal intervals plotted over test predictions
q90 = np.quantile(val_residuals, 0.90)
q95 = np.quantile(val_residuals, 0.95)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(test.index, y_test, 'k-', linewidth=1.5, label='Actual')
ax.plot(test.index, xgb_pred, 'b-', linewidth=1.2, alpha=0.8, label='XGBoost prediction')
ax.fill_between(test.index, xgb_pred-q95, xgb_pred+q95,
                alpha=0.15, color='blue', label='95% conformal')
ax.fill_between(test.index, xgb_pred-q90, xgb_pred+q90,
                alpha=0.25, color='blue', label='90% conformal')
ax.set_xlabel('Date')
ax.set_ylabel('BTC price (USD)')
ax.set_title('XGBoost predictions with split-conformal intervals')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_conformal_intervals.png', dpi=150)
plt.show()


## Phase 7 -- Interpretability: TreeSHAP + ablation study

In [ ]:
# TreeSHAP on XGBoost
print('Computing SHAP values...')
explainer = shap.TreeExplainer(xgb_final)
shap_values = explainer.shap_values(X_test)
mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
shap_df['rank'] = range(1, len(shap_df) + 1)

top10 = shap_df.head(10)
print('\nTable 12 -- Top-10 features by mean |SHAP| (XGBoost on test set)')
print(top10[['rank', 'feature', 'mean_abs_shap']].to_string(index=False,
      formatters={'mean_abs_shap': '{:,.0f}'.format}))
shap_df.to_csv(RES_DIR / 'table12_shap.csv', index=False)


In [ ]:
# Figure 5 -- SHAP bar chart
fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#3498db' if 'price' in f or f in ['open','high','low','close','ema_7','ema_30'] else '#e74c3c' for f in top10['feature']]
ax.barh(top10['feature'][::-1], top10['mean_abs_shap'][::-1], color=colors[::-1])
ax.set_xlabel('Mean |SHAP| value')
ax.set_title('Top-10 features by mean absolute SHAP value (XGBoost)')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig5_shap.png', dpi=150)
plt.show()


In [ ]:
# Ablation study -- Table 13
def fit_xgb_subset(feature_subset, label):
    idx = [FEATURE_COLS.index(f) for f in feature_subset]
    Xtr = X_trainval[:, idx]
    Xte = X_test[:, idx]
    m = xgb.XGBRegressor(learning_rate=best['lr'], max_depth=int(best['depth']),
                        n_estimators=int(best['n_est']),
                        random_state=RANDOM_SEED, n_jobs=-1, verbosity=0)
    m.fit(Xtr, y_trainval)
    pred = m.predict(Xte)
    rmse = float(np.sqrt(((y_test - pred)**2).mean()))
    mape = float((np.abs((y_test - pred)/y_test).mean()) * 100)
    return {'Configuration': label, 'd': len(feature_subset),
            'RMSE (USD)': rmse, 'MAPE (%)': mape}

top5 = shap_df.head(5)['feature'].tolist()
non_price = [f for f in FEATURE_COLS if f not in
             ['open', 'high', 'low', 'close', 'volume', 'market_cap',
              'ema_7', 'ema_30']]

ablation = pd.DataFrame([
    fit_xgb_subset(FEATURE_COLS, 'Full set (20 features)'),
    fit_xgb_subset(top5, 'Top-5 SHAP'),
    fit_xgb_subset(non_price, 'Non-price only')
])
print('Table 13 -- Feature ablation study (XGBoost retrained on each subset)')
print(ablation.to_string(index=False, formatters={
    'RMSE (USD)': '${:,.0f}'.format,
    'MAPE (%)': '{:.2f}'.format
}))
ablation.to_csv(RES_DIR / 'table13_ablation.csv', index=False)


## Phase 8 -- Robustness: multi-horizon and regime-specific analysis

In [ ]:
# Multi-horizon
horizons_results = []
for h in [1, 7, 30]:
    # Redefine target = price h days ahead
    target_h_col = f'target_h{h}'
    df_h = df.copy()
    df_h[target_h_col] = df_h['close'].shift(-h)
    df_h = df_h.dropna(subset=[target_h_col])

    train_h = df_h.loc[:TRAIN_END]
    val_h = df_h.loc[pd.Timestamp(TRAIN_END)+pd.Timedelta(days=1):VAL_END]
    test_h = df_h.loc[pd.Timestamp(VAL_END)+pd.Timedelta(days=1):TEST_END]

    sc_h = StandardScaler()
    Xtr_h = sc_h.fit_transform(train_h[FEATURE_COLS])
    Xv_h = sc_h.transform(val_h[FEATURE_COLS])
    Xte_h = sc_h.transform(test_h[FEATURE_COLS])
    ytr_h = train_h[target_h_col].values
    yv_h = val_h[target_h_col].values
    yte_h = test_h[target_h_col].values
    Xtv_h = np.vstack([Xtr_h, Xv_h])
    ytv_h = np.concatenate([ytr_h, yv_h])

    m = xgb.XGBRegressor(learning_rate=best['lr'], max_depth=int(best['depth']),
                        n_estimators=int(best['n_est']),
                        random_state=RANDOM_SEED, n_jobs=-1, verbosity=0)
    m.fit(Xtv_h, ytv_h)
    pred = m.predict(Xte_h)
    rmse_h = float(np.sqrt(((yte_h - pred)**2).mean()))
    mape_h = float((np.abs((yte_h - pred)/yte_h).mean()) * 100)
    horizons_results.append({'Horizon': f'{h} day{"s" if h > 1 else ""}',
                             'RMSE (USD)': rmse_h, 'MAPE (%)': mape_h})

horizons_df = pd.DataFrame(horizons_results)
print('Table 14 -- XGBoost multi-horizon performance')
print(horizons_df.to_string(index=False, formatters={
    'RMSE (USD)': '${:,.0f}'.format,
    'MAPE (%)': '{:.2f}'.format
}))
horizons_df.to_csv(RES_DIR / 'table14_horizons.csv', index=False)


In [ ]:
# Regime analysis
regimes = {
    'Bull (Q1 2022)': ('2022-01-01', '2022-03-31'),
    'Bear (Q2-Q4 2022)': ('2022-04-01', '2022-12-31'),
    'Recovery (2023)': ('2023-01-01', '2023-12-30')
}

regime_rows = []
for reg_name, (start, end) in regimes.items():
    mask = (test.index >= start) & (test.index <= end)
    yt = y_test[mask]
    Pt = P_test[mask]
    row = {'Regime': reg_name, 'n': int(mask.sum()),
           'Mean price': float(Pt.mean())}

    for model_name, pred in predictions.items():
        pred_r = pred[mask]
        rmse_r = float(np.sqrt(((yt - pred_r)**2).mean()))
        s_r, _ = long_short_sharpe(Pt, pred_r, 0)
        row[f'{model_name} RMSE'] = rmse_r
        row[f'{model_name} Sharpe'] = s_r

    regime_rows.append(row)

regime_df = pd.DataFrame(regime_rows)
# Print RMSE for ALL models (v15 displayed four; the computation always covered all seven)
print('Table 15 -- Per-regime RMSE, all models')
rmse_cols = [c for c in regime_df.columns if c.endswith(' RMSE')]
show = regime_df[['Regime', 'n', 'Mean price'] + rmse_cols].copy()
fmt = {c: '${:,.0f}'.format for c in ['Mean price'] + rmse_cols}
print(show.to_string(index=False, formatters=fmt))

# Per-regime RMSEs must pool back to the overall test RMSE.
print('\nPooling check  sqrt(sum(n_i * RMSE_i^2) / sum(n_i)):')
_n = regime_df['n'].values
for _c in rmse_cols:
    _pooled = float(np.sqrt((_n * regime_df[_c].values ** 2).sum() / _n.sum()))
    _model = _c.replace(' RMSE', '')
    _ref = float(np.sqrt(((y_test - predictions[_model]) ** 2).mean()))
    _flag = 'OK ' if abs(_pooled - _ref) < 1.0 else '** MISMATCH **'
    print(f'  {_model:<20s} pooled ${_pooled:>8,.0f}   full-sample ${_ref:>8,.0f}   {_flag}')
regime_df.to_csv(RES_DIR / 'table15_regimes.csv', index=False)


### Headline summary -- Table 7 (main results) consolidating all metrics

In [ ]:
# Build the headline Table 7
main_table = pd.DataFrame(all_metrics).rename(columns={
    'model': 'Model', 'rmse': 'RMSE (USD)', 'mae': 'MAE (USD)',
    'mape': 'MAPE (%)', 'r2': 'R^2', 'da': 'DA (%)',
    'sharpe_0bp': 'Sharpe (0 bp)'
})

# Add PT stats
pt_lookup = pt_df.set_index('Model')[['PT z', 'PT p']].to_dict('index')
main_table['PT z'] = main_table['Model'].map(lambda m: pt_lookup[m]['PT z'])
main_table['PT p'] = main_table['Model'].map(lambda m: pt_lookup[m]['PT p'])

# Drop n_ties from display
main_table = main_table[['Model', 'RMSE (USD)', 'MAE (USD)', 'MAPE (%)', 'R^2',
                          'DA (%)', 'PT z', 'PT p', 'Sharpe (0 bp)']]

print('Table 7 -- Main test-set performance')
print(main_table.to_string(index=False, formatters={
    'RMSE (USD)': '${:,.0f}'.format,
    'MAE (USD)': '${:,.0f}'.format,
    'MAPE (%)': '{:.2f}'.format,
    'R^2': '{:.3f}'.format,
    'DA (%)': '{:.2f}'.format,
    'PT z': '{:+.2f}'.format,
    'PT p': '{:.3f}'.format,
    'Sharpe (0 bp)': '{:+.3f}'.format
}))
print(f'\nBuy & Hold reference Sharpe (0 bp): {bh_sharpe:+.3f}')
main_table.to_csv(RES_DIR / 'table7_main_results.csv', index=False)


## Phase 9 -- [Requires Internet] B1: On-chain features enrichment

**Critique addressed:** R3 (Deep Learning), R9 (Feature Engineering), R10 (Editor)

This phase fetches on-chain Bitcoin metrics from the **CoinMetrics community API**
(free, no key required) and integrates them into the XGBoost pipeline. We then
compare:
- v14 baseline (20 features, RMSE around $1,590)
- v15 enriched (20 + 5 on-chain features)

Selected on-chain features:
- `AdrActCnt` -- active addresses (proxy for network adoption)
- `HashRate` -- network hash rate (proxy for mining security)
- `TxCnt` -- transaction count (proxy for usage)
- `RevUSD` -- miner revenue (USD)
- `DiffMean` -- network difficulty (mining difficulty)

> [WARNING] This cell requires an active internet connection. If the API is unreachable,
> the cell will skip and the rest of the notebook continues.

In [ ]:
# B1 -- Fetch on-chain features from CoinMetrics community API
ONCHAIN_AVAILABLE = False

try:
    # Method 1: use coinmetrics-api-client if installed
    try:
        from coinmetrics.api_client import CoinMetricsClient
        client = CoinMetricsClient()
        data = client.get_asset_metrics(
            assets='btc',
            metrics='AdrActCnt,HashRate,TxCnt,RevUSD,DiffMean',
            frequency='1d',
            start_time='2017-01-01',
            end_time='2023-12-31'
        ).to_dataframe()
        data['time'] = pd.to_datetime(data['time'])
        data = data.rename(columns={'time': 'date'}).set_index('date').sort_index()
        data.index = data.index.tz_localize(None)
        onchain_df = data[['AdrActCnt', 'HashRate', 'TxCnt', 'RevUSD', 'DiffMean']].copy()
        for c in onchain_df.columns:
            onchain_df[c] = pd.to_numeric(onchain_df[c], errors='coerce')
        ONCHAIN_AVAILABLE = True
        print(f'[OK] On-chain data loaded via coinmetrics-api-client: {onchain_df.shape}')
    except ImportError:
        # Method 2: direct CSV download from CoinMetrics GitHub
        print('coinmetrics-api-client not installed; trying direct CSV download...')
        url = 'https://raw.githubusercontent.com/coinmetrics/data/master/csv/btc.csv'
        raw = pd.read_csv(url, parse_dates=['time'])
        raw = raw.rename(columns={'time': 'date'}).set_index('date').sort_index()
        cols = [c for c in ['AdrActCnt', 'HashRate', 'TxCnt', 'RevUSD', 'DiffMean'] if c in raw.columns]
        onchain_df = raw[cols].copy()
        ONCHAIN_AVAILABLE = True
        print(f'[OK] On-chain data loaded via direct CSV: {onchain_df.shape}')

except Exception as e:
    print(f'[WARNING]  On-chain fetch failed: {e}')
    print('Skipping Phase 9. Run this notebook with internet access to populate this section.')

if ONCHAIN_AVAILABLE:
    # Restrict to study period and handle missing
    onchain_df = onchain_df.loc['2017-01-01':'2023-12-31'].ffill().bfill()
    print(f'On-chain features after cleaning: {onchain_df.shape}')
    print(f'Date range: {onchain_df.index.min().date()} -> {onchain_df.index.max().date()}')
    print('\nSummary statistics:')
    print(onchain_df.describe().T[['mean', 'std', 'min', 'max']].round(2))


In [ ]:
# B1 -- Retrain XGBoost with on-chain features added
if ONCHAIN_AVAILABLE:
    # Merge with existing features
    df_v2 = df.copy().join(onchain_df, how='left').ffill().bfill()

    # Apply log-transform to long-tailed on-chain metrics
    for c in ['HashRate', 'TxCnt', 'RevUSD', 'DiffMean']:
        if c in df_v2.columns:
            df_v2[f'log_{c.lower()}'] = np.log1p(df_v2[c])
            df_v2 = df_v2.drop(columns=[c])

    onchain_features_added = ['AdrActCnt', 'log_hashrate', 'log_txcnt',
                              'log_revusd', 'log_diffmean']
    onchain_features_added = [c for c in onchain_features_added if c in df_v2.columns]
    FEATURE_COLS_V2 = FEATURE_COLS + onchain_features_added
    print(f'Total features now: {len(FEATURE_COLS_V2)}')

    # Re-split
    train_v2 = df_v2.loc[:TRAIN_END]
    val_v2 = df_v2.loc[pd.Timestamp(TRAIN_END)+pd.Timedelta(days=1):VAL_END]
    test_v2 = df_v2.loc[pd.Timestamp(VAL_END)+pd.Timedelta(days=1):TEST_END]

    sc_v2 = StandardScaler()
    Xtr_v2 = sc_v2.fit_transform(train_v2[FEATURE_COLS_V2])
    Xv_v2 = sc_v2.transform(val_v2[FEATURE_COLS_V2])
    Xte_v2 = sc_v2.transform(test_v2[FEATURE_COLS_V2])
    ytr_v2 = train_v2[TARGET_COL].values
    yv_v2 = val_v2[TARGET_COL].values
    yte_v2 = test_v2[TARGET_COL].values
    Xtv_v2 = np.vstack([Xtr_v2, Xv_v2])
    ytv_v2 = np.concatenate([ytr_v2, yv_v2])

    xgb_v2 = xgb.XGBRegressor(
        learning_rate=best['lr'], max_depth=int(best['depth']),
        n_estimators=int(best['n_est']),
        random_state=RANDOM_SEED, n_jobs=-1, verbosity=0
    )
    xgb_v2.fit(Xtv_v2, ytv_v2)
    xgb_v2_pred = xgb_v2.predict(Xte_v2)

    m_v2 = compute_metrics(yte_v2, xgb_v2_pred, 'XGBoost + on-chain (B1)')
# Retrieve XGBoost baseline metrics from all_metrics (the dict was overwritten earlier)
    xgb_baseline = next(x for x in all_metrics if x['model'] == 'XGBoost')

    print('\n=== B1 Result: XGBoost with on-chain features ===')
    print(f'RMSE          v14 baseline: ${xgb_baseline["rmse"]:,.0f}')
    print(f'RMSE          v15 + B1:     ${m_v2["rmse"]:,.0f}')
    print(f'Delta RMSE                  : ${m_v2["rmse"] - xgb_baseline["rmse"]:+,.0f}')
    print(f'Delta DA (pp)              : {m_v2["da"] - xgb_baseline["da"]:+.2f}')

    # SHAP on B1
    expl_v2 = shap.TreeExplainer(xgb_v2)
    sv_v2 = expl_v2.shap_values(Xte_v2)
    shap_v2_df = pd.DataFrame({
        'feature': FEATURE_COLS_V2,
        'mean_abs_shap': np.abs(sv_v2).mean(axis=0)
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    shap_v2_df['rank'] = range(1, len(shap_v2_df) + 1)
    print('\nTop-10 features in v15 (with on-chain):')
    print(shap_v2_df.head(10).to_string(index=False, formatters={
        'mean_abs_shap': '{:,.0f}'.format
    }))
    shap_v2_df.to_csv(RES_DIR / 'shap_v15_with_onchain.csv', index=False)

    # DM test v15 vs v14
    e_v14 = errors['XGBoost']
    e_v15 = (yte_v2 - xgb_v2_pred)**2
    dm_v15, p_v15 = diebold_mariano_hln(e_v14, e_v15)
    print(f'\nDM test (v14 vs v15 + on-chain): DM = {dm_v15:+.2f}, p = {p_v15:.3e}')
    if abs(dm_v15) > 1.96 and p_v15 < 0.05:
        if dm_v15 > 0:
            print('=> v15 with on-chain features is statistically more accurate.')
        else:
            print('=> v14 baseline is statistically more accurate.')
    else:
        print('=> No statistically significant difference at the 5% level.')

else:
    print('Phase 9 skipped (no internet / API failure)')


## Phase 10 -- [Requires Internet] B4: Daily Google Trends via pytrends

**Critique addressed:** R4 (Data Quality), R5 (Crypto Insider)

The v14 dataset uses Google Trends data interpolated from native monthly frequency
to daily, which destroys high-frequency variation and underestimates the
contribution of attention signals. This phase fetches **native daily** Google
Trends data via `pytrends` (8-month chunks, then renormalised to a single 0-100 scale).

> [WARNING] Requires internet access. Google Trends rate-limits aggressive fetching;
> we use 5-second sleep between chunks.

> [WARNING] pytrends API can return empty responses if Google Trends is being scraped
> heavily; the cell handles this gracefully.

In [ ]:
# B4 -- Fetch daily Google Trends
DAILY_TRENDS_AVAILABLE = False

try:
    from pytrends.request import TrendReq
except ImportError:
    print('pytrends not installed. To run this section: pip install pytrends')
    raise SystemExit
else:
    print('pytrends installed')

try:
    pytrends = TrendReq(hl='en-US', tz=360, retries=3)
    chunks = []
    current = pd.Timestamp('2017-01-01')
    end = pd.Timestamp('2023-12-31')
    chunk_months = 8

    while current < end:
        chunk_end = min(current + pd.DateOffset(months=chunk_months), end)
        timeframe = f'{current.strftime("%Y-%m-%d")} {chunk_end.strftime("%Y-%m-%d")}'
        print(f'   Fetching {timeframe}...')
        pytrends.build_payload(['bitcoin'], cat=0, timeframe=timeframe, geo='', gprop='')
        chunk = pytrends.interest_over_time()
        if not chunk.empty:
            chunks.append(chunk[['bitcoin']])
        current = chunk_end + pd.Timedelta(days=1)
        time.sleep(5)

    if chunks:
        gt_daily = pd.concat(chunks)
        gt_daily = gt_daily[~gt_daily.index.duplicated(keep='first')]
        gt_daily = gt_daily.rename(columns={'bitcoin': 'google_trends_raw'})
        # Renormalize globally
        gt_daily['google_trends_daily'] = (
            100 * gt_daily['google_trends_raw'] / gt_daily['google_trends_raw'].max()
        )
        gt_daily.to_csv(RES_DIR / 'btc_google_trends_daily.csv')
        DAILY_TRENDS_AVAILABLE = True
        print(f'\n[OK] Fetched {len(gt_daily)} daily observations')
    else:
        print('[WARNING]  pytrends returned empty results')

except Exception as e:
    print(f'[WARNING]  Daily Trends fetch failed: {e}')
    print('Skipping Phase 10. Common reasons: rate-limiting, network issues.')


In [ ]:
# B4 -- Compare interpolated (v14) vs daily (v15)
if DAILY_TRENDS_AVAILABLE:
    gt_v14 = df['google_trends']
    gt_v15 = gt_daily['google_trends_daily']

    # Align
    common_idx = gt_v14.index.intersection(gt_v15.index)
    gt_v14_a = gt_v14.loc[common_idx]
    gt_v15_a = gt_v15.loc[common_idx]

    corr_p, _ = stats.pearsonr(gt_v14_a, gt_v15_a)
    print(f'Pearson correlation interpolated vs daily: {corr_p:.3f}')
    print(f'Std (interpolated, v14): {gt_v14_a.std():.2f}')
    print(f'Std (daily,        v15): {gt_v15_a.std():.2f}')
    print(f'Std ratio v15/v14:       {gt_v15_a.std()/gt_v14_a.std():.2f}x (>1 = daily has more variance)')

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(common_idx, gt_v14_a, label='Interpolated (v14, monthly->daily)', linewidth=1.0, alpha=0.8)
    ax.plot(common_idx, gt_v15_a, label='Daily native (v15, via pytrends)', linewidth=1.0, alpha=0.8)
    ax.set_xlabel('Date'); ax.set_ylabel('Google Trends (0-100)')
    ax.set_title('Google Trends BTC: interpolated monthly vs native daily')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'figB4_trends_comparison.png', dpi=150)
    plt.show()

    # Retrain XGBoost with daily trends replacing interpolated
    df_b4 = df.copy()
    df_b4['google_trends'] = gt_v15.reindex(df_b4.index).ffill().bfill()
    train_b4 = df_b4.loc[:TRAIN_END]
    val_b4 = df_b4.loc[pd.Timestamp(TRAIN_END)+pd.Timedelta(days=1):VAL_END]
    test_b4 = df_b4.loc[pd.Timestamp(VAL_END)+pd.Timedelta(days=1):TEST_END]
    sc_b4 = StandardScaler()
    Xtr_b4 = sc_b4.fit_transform(train_b4[FEATURE_COLS])
    Xv_b4 = sc_b4.transform(val_b4[FEATURE_COLS])
    Xte_b4 = sc_b4.transform(test_b4[FEATURE_COLS])
    Xtv_b4 = np.vstack([Xtr_b4, Xv_b4])
    ytv_b4 = np.concatenate([train_b4[TARGET_COL].values, val_b4[TARGET_COL].values])
    yte_b4 = test_b4[TARGET_COL].values

    xgb_b4 = xgb.XGBRegressor(
        learning_rate=best['lr'], max_depth=int(best['depth']),
        n_estimators=int(best['n_est']),
        random_state=RANDOM_SEED, n_jobs=-1, verbosity=0
    )
    xgb_b4.fit(Xtv_b4, ytv_b4)
    xgb_b4_pred = xgb_b4.predict(Xte_b4)
    m_b4 = compute_metrics(yte_b4, xgb_b4_pred, 'XGBoost + daily trends (B4)')

    # Retrieve XGBoost baseline metrics from all_metrics
    xgb_baseline = next(x for x in all_metrics if x['model'] == 'XGBoost')

    print('\n=== B4 Result: XGBoost with daily Google Trends ===')
    print(f'RMSE v14 baseline   : ${xgb_baseline["rmse"]:,.0f}')
    print(f'RMSE v15 + B4       : ${m_b4["rmse"]:,.0f}')
    print(f'Delta RMSE          : ${m_b4["rmse"] - xgb_baseline["rmse"]:+,.0f}')
    print(f'Delta MAPE (pp)     : {m_b4["mape"] - xgb_baseline["mape"]:+.2f}')
    print(f'Delta DA (pp)       : {m_b4["da"] - xgb_baseline["da"]:+.2f}')

    # New SHAP rank of google_trends
    expl_b4 = shap.TreeExplainer(xgb_b4)
    sv_b4 = expl_b4.shap_values(Xte_b4)
    shap_b4 = pd.DataFrame({
        'feature': FEATURE_COLS,
        'mean_abs_shap': np.abs(sv_b4).mean(axis=0)
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    shap_b4['rank'] = range(1, len(shap_b4) + 1)

    gt_rank_v14 = int(shap_df.loc[shap_df['feature']=='google_trends', 'rank'].iloc[0])
    gt_rank_b4 = int(shap_b4.loc[shap_b4['feature']=='google_trends', 'rank'].iloc[0])
    print(f'\ngoogle_trends SHAP rank, v14 (interpolated): #{gt_rank_v14}')
    print(f'google_trends SHAP rank, B4 (daily native):  #{gt_rank_b4}')
    if gt_rank_b4 < gt_rank_v14:
        print(f'=> Daily Google Trends moved up by {gt_rank_v14 - gt_rank_b4} ranks -> confirms')
        print('   the v14 SHAP rank was an artifact of the interpolation.')
    else:
        print('=> No improvement: Google Trends remains a weak signal at daily frequency.')
else:
    print('Phase 10 skipped (no internet / pytrends issue)')

## Phase 11 -- Final summary and submission artefacts

This final section consolidates:
- All test-set metrics for the 7 models
- Headline statistical conclusions
- Summary of v14 vs v15 enrichments (if Phases 9/10 ran)
- Reproducibility manifest (random seeds, library versions, runtime)

In [ ]:
# Reproducibility manifest
manifest = {
    'random_seed': RANDOM_SEED,
    'seeds_for_dl': SEEDS,
    'split': {
        'train_end': TRAIN_END,
        'val_end': VAL_END,
        'test_end': TEST_END,
        'n_train': len(train), 'n_val': len(val), 'n_test': len(test)
    },
    'xgb_best_config': {
        'learning_rate': float(best['lr']),
        'max_depth': int(best['depth']),
        'n_estimators': int(best['n_est'])
    },
    'phase_9_b1_onchain': bool(ONCHAIN_AVAILABLE),
    'phase_10_b4_daily_trends': bool(DAILY_TRENDS_AVAILABLE),
    'library_versions': {
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'xgboost': xgb.__version__,
        'tensorflow': tf.__version__,
        'shap': shap.__version__,
        'sklearn': __import__('sklearn').__version__,
        'statsmodels': sm.__version__,
        'scipy': stats.__name__.split('.')[0]
    }
}
with open(RES_DIR / 'manifest_v15.json', 'w') as f:
    json.dump(manifest, f, indent=2, default=str)
print('Manifest saved to results/manifest_v15.json')
print(json.dumps(manifest, indent=2, default=str))


In [ ]:
# Final headline summary
print('='*72)
print('NOTEBOOK v15 -- FINAL SUMMARY')
print('='*72)
print()
print('### Test-set performance (n = 729 days, Jan 2022 – Dec 2023) ###\n')
print(main_table.to_string(index=False, formatters={
    'RMSE (USD)': '${:,.0f}'.format,
    'MAE (USD)': '${:,.0f}'.format,
    'MAPE (%)': '{:.2f}'.format,
    'R^2': '{:.3f}'.format,
    'DA (%)': '{:.2f}'.format,
    'PT z': '{:+.2f}'.format,
    'PT p': '{:.3f}'.format,
    'Sharpe (0 bp)': '{:+.3f}'.format
}))
print(f'\nBuy & Hold benchmark Sharpe: {bh_sharpe:+.3f}')
print()
print('### Headline conclusions (aligned with manuscript v15) ###\n')
xgb_metrics = next(m for m in all_metrics if m['model']=='XGBoost')
_won = dm_df[dm_df['Conclusion'].str.contains('XGBoost wins')]
_max_bonf = float(_won['p (Bonf.)'].max())
print(f'1. XGBoost is the best learned model by RMSE (${xgb_metrics["rmse"]:,.0f}),')
print('   statistically dominating every learned challenger under HLN-corrected DM tests')
print(f'   (all Bonferroni-adjusted p <= {_max_bonf:.3g}).')
print()
_da = np.array([m['da'] for m in all_metrics], dtype=float)
_n_sig_raw = int((pt_df['PT p'] < 0.05).sum())
_se_da = 100 * 0.5 / np.sqrt(len(y_test) - 1)
_mde = 50 + 1.96 * _se_da
print(f'2. THE DIRECTIONAL ACCURACY PITFALL: {len(_da) - _n_sig_raw}/{len(_da)} models fail to')
print(f'   reject the PT no-skill null at the raw 5% level (min p = {pt_df["PT p"].min():.3f}).')
print(f'   DA spans {_da.min():.2f}-{_da.max():.2f}%.')
print(f'   Design resolution: SE(DA) = {_se_da:.2f} pp under the null, so the test')
print(f'   detects DA above ~{_mde:.1f}% at alpha = 0.05 (two-tailed). This is an')
print(f'   absence of skill at that resolution, not a demonstration of zero skill.')
print()
print(f'3. Spearman rho between DA and Sharpe (n=7) = {spearman_r:+.3f}, p = {spearman_p:.3f}:')
print('   no monotonic relationship between DA and economic utility.')
print()
print(f'4. At 10 bp transaction costs, no learned model robustly beats Buy & Hold')
print(f'   (B&H Sharpe = {bh_sharpe:+.3f}).')
print()
if ONCHAIN_AVAILABLE:
    print(f'5. [B1] Adding on-chain features changes XGBoost RMSE by '
          f'${m_v2["rmse"] - xgb_metrics["rmse"]:+,.0f} (statistically '
          f'{"significant" if abs(dm_v15) > 1.96 else "non-significant"}).')
if DAILY_TRENDS_AVAILABLE:
    print(f'6. [B4] Replacing interpolated with daily Google Trends moves the')
    print(f'   google_trends SHAP rank from #{gt_rank_v14} to #{gt_rank_b4}.')
print()
print('### Reproducibility ###\n')
print(f'Random seed: {RANDOM_SEED}')
print(f'Deep learning seeds: {SEEDS}')
print(f'All numerical results saved to: {RES_DIR}/')
print(f'All figures saved to:           {FIG_DIR}/')
print()
print('='*72)
print('=== END OF NOTEBOOK ===')
print('='*72)


In [ ]:
# ============================================================
# PATCH : 6 MISSING FIGURES FOR THE MANUSCRIPT
# ============================================================
# This patch generates figures 1, 2, 3, 5, 8 and 9 referenced
# in the LaTeX manuscript but missing from the original notebook.
#
# HOW TO USE:
# - Run this AFTER all phases of the main notebook have completed
# - It uses `predictions`, `errors`, `dm_df`, `shap_df`, `ablation`,
#   `horizons_df`, `regime_df`, `test`, `df`, `y_test` already in memory
# - Adds 6 PNG files to your figures/ directory
# ============================================================

print("="*60)
print("GENERATING MISSING FIGURES (1, 2, 3, 5, 8, 9)")
print("="*60)

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# === FIGURE 1: All model predictions vs actual ===
print("\n[1/6] Generating fig1_predictions.png ...")
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(test.index, y_test, 'k-', linewidth=2.0, label='Actual', alpha=0.9)
colors = sns.color_palette('Set2', n_colors=len(predictions))
for (name, pred), color in zip(predictions.items(), colors):
    ax.plot(test.index, pred, linewidth=1.0, alpha=0.7, label=name, color=color)
ax.set_xlabel('Date')
ax.set_ylabel('BTC closing price (USD)')
ax.set_title('Model predictions vs. actual BTC closing price (test set 2022-2023)')
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_predictions.png', dpi=150)
plt.show()
print("  -> fig1_predictions.png saved")

# === FIGURE 2: Stationarity diagnostics (close vs log_return) ===
print("\n[2/6] Generating fig2_stationarity.png ...")
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(df.index, df['close'], 'b-', linewidth=0.7)
axes[0].set_ylabel('BTC closing price (USD)')
axes[0].set_title('Top: closing price (non-stationary, ADF p=0.514).  '
                  'Bottom: log returns (stationary, ADF p<0.001)')
axes[0].grid(True, alpha=0.3)
axes[1].plot(df.index, df['log_return'], 'r-', linewidth=0.5, alpha=0.7)
axes[1].axhline(0, color='k', linewidth=0.5)
axes[1].set_ylabel('Daily log return')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_stationarity.png', dpi=150)
plt.show()
print("  -> fig2_stationarity.png saved")

# === FIGURE 3: DM forest plot vs XGBoost ===
print("\n[3/6] Generating fig3_dm_forest.png ...")
fig, ax = plt.subplots(figsize=(9, 5))
y_pos = np.arange(len(dm_df))
colors_dm = ['red' if dm < 0 else 'green' for dm in dm_df['DM stat']]
ax.barh(y_pos, dm_df['DM stat'], color=colors_dm, alpha=0.7)
ax.axvline(0, color='k', linewidth=0.5)
# Bonferroni-adjusted threshold: alpha = 0.05/6 = 0.0083 -> two-tailed |z| = 2.64.
# This is the level the manuscript tests at; v15 drew the unadjusted 1.96.
_z_bonf = float(stats.norm.ppf(1 - (0.05 / len(dm_df)) / 2))
ax.axvline(_z_bonf, color='gray', linestyle='--', linewidth=0.8,
           label=f'|z|={_z_bonf:.2f} (Bonferroni, alpha={0.05/len(dm_df):.4f})')
ax.axvline(-_z_bonf, color='gray', linestyle='--', linewidth=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(dm_df['Challenger'])
ax.set_xlabel('DM statistic (HLN-corrected, positive => XGBoost wins)')
ax.set_title('Diebold-Mariano forest plot vs. XGBoost reference')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_dm_forest.png', dpi=150)
plt.show()
print("  -> fig3_dm_forest.png saved")

# === FIGURE 5: Multi-horizon degradation ===
print("\n[5/6] Generating fig5_horizons.png ...")
fig, ax1 = plt.subplots(figsize=(9, 5))
horizons_n = [1, 7, 30]
# horizons_df has columns: Horizon, RMSE (USD), MAPE (%)
ax1.plot(horizons_n, horizons_df['RMSE (USD)'], 'b-o', linewidth=2.0, markersize=10, label='RMSE')
ax1.set_xlabel('Prediction horizon (days)')
ax1.set_ylabel('RMSE (USD)', color='b')
ax1.tick_params(axis='y', labelcolor='b')
ax1.grid(True, alpha=0.3)
ax2 = ax1.twinx()
ax2.plot(horizons_n, horizons_df['MAPE (%)'], 'r-s', linewidth=2.0, markersize=10, label='MAPE')
ax2.set_ylabel('MAPE (%)', color='r')
ax2.tick_params(axis='y', labelcolor='r')
ax1.set_title('XGBoost performance degradation with prediction horizon')
ax1.set_xticks(horizons_n)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig5_horizons.png', dpi=150)
plt.show()
print("  -> fig5_horizons.png saved")

# === FIGURE 8: Feature ablation study ===
print("\n[6/6] Generating fig8_ablation.png ...")
fig, ax = plt.subplots(figsize=(8, 5))
# ablation has columns: Configuration, d, RMSE (USD), MAPE (%)
configs = ablation['Configuration']
rmses = ablation['RMSE (USD)']
mapes = ablation['MAPE (%)']
colors_abl = ['#3498db', '#f39c12', '#e74c3c']
bars = ax.bar(configs, rmses, color=colors_abl, alpha=0.8)
for i, (bar, m_val) in enumerate(zip(bars, mapes)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(rmses)*0.02,
            f'${rmses.iloc[i]:,.0f}\n({m_val:.1f}% MAPE)',
            ha='center', fontsize=10)
ax.set_ylabel('RMSE (USD)')
ax.set_title('XGBoost feature-ablation study')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig8_ablation.png', dpi=150)
plt.show()
print("  -> fig8_ablation.png saved")

# === FIGURE 9: Regime RMSE heatmap ===
print("\n[6/6] Generating fig9_regime_heatmap.png ...")
# regime_df columns include '<Model> RMSE' for each model
regime_rmse_cols = [c for c in regime_df.columns if 'RMSE' in c and 'Sharpe' not in c]
regime_matrix = regime_df[regime_rmse_cols].T
regime_matrix.columns = regime_df['Regime']
regime_matrix.index = [c.replace(' RMSE', '') for c in regime_matrix.index]

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(regime_matrix, annot=True, fmt=',.0f', cmap='YlOrRd',
            cbar_kws={'label': 'RMSE (USD)'}, ax=ax,
            linewidths=0.5, linecolor='white')
ax.set_title('RMSE (USD) by model x market regime')
ax.set_xlabel('Market regime')
ax.set_ylabel('Model')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig9_regime_heatmap.png', dpi=150)
plt.show()
print("  -> fig9_regime_heatmap.png saved")

print("\n" + "="*60)
print("All 6 missing figures generated successfully!")
print("="*60)
print(f"Figures saved to: {FIG_DIR}/")
print("\nList of figures in directory:")
import os
for f in sorted(os.listdir(FIG_DIR)):
    if f.endswith('.png'):
        size_kb = os.path.getsize(FIG_DIR / f) / 1024
        print(f"  {f}: {size_kb:.1f} KB")

In [ ]:
# Align figure filenames with the LaTeX manuscript.
# v15 skipped any target that already existed, so stale PNGs from earlier runs
# survived into the build folder. This version always overwrites.
import shutil

renames = {
    'fig7_tc_sensitivity.png': 'fig11_tc_sensitivity.png',
    'fig5_shap.png': 'fig7_shap_top10.png',
}

for old, new in renames.items():
    old_path, new_path = FIG_DIR / old, FIG_DIR / new
    if old_path.exists():
        shutil.copyfile(old_path, new_path)   # overwrite unconditionally
        print(f'copied (overwrite): {old} -> {new}')
    else:
        print(f'MISSING SOURCE    : {old} -- re-run the cell that generates it')

print('\nFigure inventory:')
for f in sorted(FIG_DIR.glob('*.png')):
    print(f'  {f.name:34s} {f.stat().st_size/1024:7.1f} KB')


## Phase 12 — Verification: figures vs tables

Guard against the failure that produced mismatched figures in earlier builds.
This cell must run clean before the artefact is archived or the manuscript
compiled.

In [ ]:
# ============================================================
# VERIFICATION -- figures vs tables, and manuscript figure inventory
# ============================================================
import numpy as np

failures = []

def check(label, cond, detail=''):
    (print(f'  OK    {label}') if cond
     else (print(f'  FAIL  {label}  {detail}'), failures.append(label)))

print('Per-regime RMSE pools to the full-sample RMSE')
_n = regime_df['n'].values
for _c in [c for c in regime_df.columns if c.endswith(' RMSE')]:
    _model = _c.replace(' RMSE', '')
    _pooled = float(np.sqrt((_n * regime_df[_c].values ** 2).sum() / _n.sum()))
    _ref = float(np.sqrt(((y_test - predictions[_model]) ** 2).mean()))
    check(f'{_model:<20s} pooled ${_pooled:,.0f} vs ${_ref:,.0f}', abs(_pooled - _ref) < 1.0)

print('\nTable values used by the figures')
check('ablation non-price RMSE matches Table 13',
      abs(float(ablation.loc[ablation["Configuration"].str.contains("Non-price"), "RMSE (USD)"].iloc[0]) - 3152) < 2)
check('all seven models present in the DA-vs-Sharpe scatter', len(all_metrics) == 7)
check('DA range reaches the walk-forward maximum',
      max(m['da'] for m in all_metrics) > 52.5,
      '(fig4 x-axis must span the full range)')
check('no model is PT-significant at 5% (fig4 must show no significance marker)',
      int((pt_df['PT p'] < 0.05).sum()) == 0)
check('Buy & Hold Sharpe equals -(Random Walk Sharpe) up to rounding',
      abs(bh_sharpe + next(m['sharpe_0bp'] for m in all_metrics if m['model'] == 'Random Walk')) < 1e-3)

print('\nManuscript figure inventory')
required = ['fig1_predictions.png', 'fig2_stationarity.png', 'fig3_dm_forest.png',
            'fig4_da_paradox.png', 'fig5_horizons.png', 'fig6_trading_cumret.png',
            'fig7_shap_top10.png', 'fig8_ablation.png', 'fig9_regime_heatmap.png',
            'fig11_tc_sensitivity.png']
for f in required:
    check(f'{f} present', (FIG_DIR / f).exists())

print('\n' + '=' * 60)
if failures:
    print(f'{len(failures)} CHECK(S) FAILED -- do not archive or compile until resolved:')
    for f in failures:
        print(f'  - {f}')
else:
    print('ALL CHECKS PASSED -- figures and tables are consistent.')
print('=' * 60)
